# Jebrex AI — Cloud GPU Resumable Training Notebook (Phase C & D)

Run `JebrexTransformer` (`125M` / `350M` with RoPE, RMSNorm, SwiGLU, GQA, Gradient Checkpointing, BF16/FP16) on Google Colab, Kaggle, or Cloud GPUs with automatic Google Drive / GCS checkpoint sync and portable SHA-256 bundles.

**Honesty & Safety Guardrails:**
- Never commit raw checkpoints (`.pt`) or `.env` secrets to GitHub.
- Train exclusively on licensed, deduplicated, leakage-checked datasets (`DATA_READY`).
- Never evaluate on the held-out test split during hyperparameter selection.

In [ ]:
# 1. Detect GPU Hardware & Mount Persistent Cloud Storage (Colab)
import os, sys, torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), '| BF16 supported:', torch.cuda.is_bf16_supported())
else:
    print('WARNING: Running on CPU. Switch Colab Runtime -> GPU (T4 / L4 / A100) for 125M-350M training.')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_ROOT = '/content/drive/MyDrive/JebrexAI/checkpoints'
except ImportError:
    CHECKPOINT_ROOT = './cloud_checkpoints'
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)
print('Checkpoint root:', CHECKPOINT_ROOT)

In [ ]:
# 2. Clone / Pull Repository (Code Only — No Large Weights in Git)
!git clone --depth 1 -b comprehensive_project_audit https://github.com/ypy999/NourAI.git jebrex_ai || (cd jebrex_ai && git pull)
%cd jebrex_ai
sys.path.insert(0, os.getcwd())

In [ ]:
# 3. Inspect Verified 125M & 350M Parameter Counts & Memory Budgets
import json
from model.jebrex_transformer import verify_preset_parameter_counts
print(json.dumps({k: {m: v[m] for m in ('parameters_millions', 'weights_bf16_mib', 'adamw_training_states_fp32_mib')} for k, v in verify_preset_parameter_counts().items()}, indent=2))

In [ ]:
# 4. Configure Cloud Training Run & Auto-Resume from Last Healthy Checkpoint
from pathlib import Path
from model.jebrex_transformer import JEBBREX_MODEL_PRESETS, JebrexTransformer
from tokenizer.bpe_tokenizer import MultilingualBPETokenizer
from datasets.efficient_loader import pack_jsonl_to_uint16_bin, build_memmap_dataloader
from training.cloud_trainer import CloudTrainConfig, CloudResumableTrainer

PRESET = 'jebrex-125m' if torch.cuda.is_available() else 'jebrex-micro'
TOKENIZER_PATH = Path('tokenizer/artifacts/jebrex_bpe_v1.json')
DATA_SPLITS_DIR = Path('data/processed/jebrex_v1_clean/splits')  # Replace with multi-billion GCS shard path for production
RUN_DIR = Path(CHECKPOINT_ROOT) / PRESET

tok = MultilingualBPETokenizer.load(TOKENIZER_PATH)
cfg = JEBBREX_MODEL_PRESETS[PRESET]
cfg.vocab_size = max(cfg.vocab_size, tok.vocab_size)
cfg.gradient_checkpointing = True

bin_dir = RUN_DIR / 'packed_bin'
pack_jsonl_to_uint16_bin(DATA_SPLITS_DIR / 'train.jsonl', bin_dir / 'train.bin', tok)
pack_jsonl_to_uint16_bin(DATA_SPLITS_DIR / 'validation.jsonl', bin_dir / 'val.bin', tok)
pack_jsonl_to_uint16_bin(DATA_SPLITS_DIR / 'test.jsonl', bin_dir / 'test.bin', tok)

train_cfg = CloudTrainConfig(
    preset=PRESET,
    batch_size=8 if torch.cuda.is_available() else 4,
    grad_accum_steps=4 if torch.cuda.is_available() else 1,
    block_size=512 if torch.cuda.is_available() else 128,
    max_steps=500 if torch.cuda.is_available() else 30,
    warmup_steps=50 if torch.cuda.is_available() else 5,
    eval_interval=50 if torch.cuda.is_available() else 15,
    save_interval=50 if torch.cuda.is_available() else 15,
    gradient_checkpointing=True,
    mixed_precision='auto',
    remote_uri=f'gdrive://JebrexAI/checkpoints/{PRESET}',
)

train_loader = build_memmap_dataloader(bin_dir / 'train.bin', train_cfg.block_size, train_cfg.batch_size, shuffle=True)
val_loader = build_memmap_dataloader(bin_dir / 'val.bin', train_cfg.block_size, train_cfg.batch_size, shuffle=False)
test_loader = build_memmap_dataloader(bin_dir / 'test.bin', train_cfg.block_size, train_cfg.batch_size, shuffle=False)

model = JebrexTransformer(cfg)
trainer = CloudResumableTrainer(model, train_cfg, RUN_DIR)
resumed = trainer.resume_latest_if_exists()
print('Resume status:', resumed)
summary = trainer.train(train_loader, val_loader)
print('Session summary:', json.dumps(summary, indent=2))

# Export verifiable portable bundle (SHA-256) to Google Drive
bundle = trainer.export_portable_bundle(RUN_DIR / 'portable_bundle')
print('Portable bundle manifest:', json.dumps(bundle, indent=2))
print('Post-training held-out test metrics:', trainer.evaluate(test_loader))